In [1]:
from pathlib import Path
import logging
import joblib
from itertools import product
import numpy as np
from tqdm import tqdm
from src.module.utils import load_config, get_peak_trough, process_meanbp
import sklearn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    classification_report,
)
import matplotlib.pyplot as plt

In [2]:
def balance_dataset(X, y):
    pos_idx, neg_idx = np.where(y == 1)[0], np.where(y == 0)[0]
    if len(pos_idx) > len(neg_idx):
        pos_idx = np.random.choice(pos_idx, size=len(neg_idx), replace=False)
    else:
        neg_idx = np.random.choice(neg_idx, size=len(pos_idx), replace=False)

    pos_X, pos_y = X[pos_idx], y[pos_idx]
    neg_X, neg_y = X[neg_idx], y[neg_idx]
    res_X = np.vstack([pos_X, neg_X])
    res_y = np.hstack([pos_y, neg_y])
    print(res_y.shape)
    logging.info(
        f"X: {res_X.shape} / y(pos/neg): {res_y.size}({sum(res_y==1)}/{sum(res_y==0)})"
    )
    return res_X, res_y


def scale_input(input):
    return input.astype(np.float32) / 200


def get_min_size(x, p, t):
    if x[p].size < x[t].size:
        min_size = x[p].size
    else:
        min_size = x[t].size
    return min_size


def summary_stat(arr):
    x = arr.reshape(-1)
    x_mean, x_std, x_median, x_min, x_max = (
        np.mean(x),
        np.std(x),
        np.median(x),
        np.min(x),
        np.max(x),
    )
    return x_mean, x_std, x_median, x_min, x_max


def slope_intercept(t, y):
    t = np.asarray(t, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    tm, ym = t.mean(), y.mean()
    dt = t - tm
    denom = (dt * dt).sum()
    if denom == 0:
        return np.nan, np.nan
    slope = (dt * (y - ym)).sum() / denom
    return slope, ym - (slope * tm)


def extract_feature(arr, config):
    feature = []
    p, t = get_peak_trough(arr, config["detection"])
    min_size = get_min_size(arr, p, t)
    mbp = process_meanbp(arr.reshape(-1), config)
    sbp, dbp, pp = arr[p], arr[t], arr[p][:min_size] - arr[t][:min_size]
    mbp = np.unique(mbp[np.where(mbp != 0)])

    sbp_stat = summary_stat(sbp)
    dbp_stat = summary_stat(dbp)
    pp_stat = summary_stat(pp)
    mbp_stat = summary_stat(mbp)
    trend_per_beat, _ = slope_intercept(np.arange(mbp.size), mbp)
    trend_per_sec = trend_per_beat * (mbp.size / 30.0)
    half = mbp.size // 2
    slope_per_beat, _ = slope_intercept(np.arange(mbp.size)[half:], mbp[half:])
    slope_per_sec = slope_per_beat * (mbp.size / 30.0)
    last_mbp = mbp[-1]

    feature.extend(
        [
            *sbp_stat,
            *dbp_stat,
            *pp_stat,
            *mbp_stat,
            trend_per_sec,
            slope_per_sec,
            last_mbp,
        ]
    )
    return feature


def brier_skill_score(y_true, y_prob):
    y_true = np.asarray(y_true, dtype=float)
    bs_model = brier_score_loss(y_true, y_prob)
    p = y_true.mean()  # 평가셋 자연 유병률
    bs_ref = p * (1.0 - p)  # 항상 p를 예측하는 무정보 모델
    return 1.0 - bs_model / bs_ref, bs_model, bs_ref

In [3]:
config = load_config("../config/conf.yaml")
src_path = Path(config["data_path"]).expanduser() / "vitaldb" / "06.train_val_test_new"
model_path = Path(config["model_path"]).expanduser() / "revision_seed42"
strategy = "hypophetversion2"

In [4]:
train_X = np.load(src_path / f"vitaldb_{strategy}_train_X.npy")
train_y = np.load(src_path / f"vitaldb_{strategy}_train_y.npy")
val_X = np.load(src_path / f"vitaldb_{strategy}_val_X.npy")
val_y = np.load(src_path / f"vitaldb_{strategy}_val_y.npy")
test_X = np.load(src_path / f"vitaldb_{strategy}_test_X.npy")
test_y = np.load(src_path / f"vitaldb_{strategy}_test_y.npy")
train_bal_X, train_bal_y = balance_dataset(train_X, train_y)
print(train_X.shape, train_y.shape)
print(train_bal_X.shape, train_bal_y.shape)
print(val_X.shape, val_y.shape)
print(test_X.shape, test_y.shape)

(98982,)
(145764, 3000, 1) (145764,)
(98982, 3000, 1) (98982,)
(37159, 3000, 1) (37159,)
(44056, 3000, 1) (44056,)


In [5]:
train_bal_X_feat = np.array(
    [extract_feature(x, config) for x in tqdm(train_bal_X, ncols=75)]
)
val_X_feat = np.array([extract_feature(x, config) for x in tqdm(val_X, ncols=75)])
test_X_feat = np.array([extract_feature(x, config) for x in tqdm(test_X, ncols=75)])
print(train_bal_X_feat.shape, val_X_feat.shape, test_X_feat.shape)

100%|██████████████████████████████| 44056/44056 [00:13<00:00, 3221.74it/s]

(98982, 23) (37159, 23) (44056, 23)


In [ ]:
# # tree-based ML은 스케일링이 필요없다.
# best = (None, -np.inf, None)
# grid = product([0.01, 0.05, 0.1], [3, 4, 6], [1, 5, 10])

# for lr, depth, mcw in grid:
#     clf = XGBClassifier(
#         n_estimators=1000,
#         learning_rate=lr,
#         max_depth=depth,
#         min_child_weight=mcw,
#         early_stopping_rounds=50,
#         random_state=config["seed"],
#     )
#     clf.fit(
#         train_bal_X_feat, train_bal_y, eval_set=[(val_X_feat, val_y)], verbose=False
#     )
#     auc = roc_auc_score(val_y, clf.predict_proba(val_X_feat)[:, 1])
#     if auc > best[1]:
#         best = ((lr, depth, mcw), auc, clf.best_iteration)

# print(best)

((0.1, 4, 5), 0.7801287554309001, 84)


In [ ]:
# lr, depth, mcw = best[0]
# clf = XGBClassifier(
#     n_estimators=1000,
#     learning_rate=lr,
#     max_depth=depth,
#     min_child_weight=mcw,
#     early_stopping_rounds=50,
#     random_state=config["seed"],
# )
# clf.fit(train_bal_X_feat, train_bal_y, eval_set=[(val_X_feat, val_y)], verbose=False)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=50,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
              max_leaves=None, min_child_weight=5, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=1000,
              n_jobs=None, num_parallel_tree=None, ...)

In [6]:
clf = XGBClassifier(random_state=config["seed"])
clf.fit(train_bal_X_feat, train_bal_y, eval_set=[(val_X_feat, val_y)], verbose=False)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [7]:
prob_val_y = clf.predict_proba(val_X_feat)[:, 1]
neg = prob_val_y[np.where(val_y == 0)]
for spe in [0.80, 0.90, 0.95]:
    cutoff = np.quantile(neg, spe)
    print(spe, cutoff)

0.8 0.586908483505249
0.9 0.7196147918701172
0.95 0.8159542202949521


In [8]:
prob_y = clf.predict_proba(test_X_feat)[:, 1]

In [9]:
auroc = roc_auc_score(test_y, prob_y)
auprc = average_precision_score(test_y, prob_y)
bss, bs, bs_null = brier_skill_score(test_y, prob_y)
print(f"AUROC: {auroc}")
print(f"AUPRC: {auprc}")
print(f"Brier={bs:.4f}, Null Brier={bs_null:.4f}, BSS={bss:.4f}")
print(classification_report(test_y, np.where(prob_y >= 0.5, 1, 0)))

AUROC: 0.7638380867913903
AUPRC: 0.6509937610444699
Brier=0.1978, Null Brier=0.2280, BSS=0.1324
              precision    recall  f1-score   support

         0.0       0.81      0.72      0.76     28569
         1.0       0.57      0.69      0.62     15487

    accuracy                           0.71     44056
   macro avg       0.69      0.70      0.69     44056
weighted avg       0.72      0.71      0.71     44056



In [10]:
clf.save_model(model_path / "xgb_baseline.json")

In [11]:
loaded_xgb = XGBClassifier()
loaded_xgb.load_model(model_path / "xgb_baseline.json")
prob_y = loaded_xgb.predict_proba(test_X_feat)[:, 1]

In [12]:
auroc = roc_auc_score(test_y, prob_y)
auprc = average_precision_score(test_y, prob_y)
bss, bs, bs_null = brier_skill_score(test_y, prob_y)
print(f"AUROC: {auroc}")
print(f"AUPRC: {auprc}")
print(f"Brier={bs:.4f}, Null Brier={bs_null:.4f}, BSS={bss:.4f}")
print(classification_report(test_y, np.where(prob_y >= 0.5, 1, 0)))

AUROC: 0.7638380867913903
AUPRC: 0.6509937610444699
Brier=0.1978, Null Brier=0.2280, BSS=0.1324
              precision    recall  f1-score   support

         0.0       0.81      0.72      0.76     28569
         1.0       0.57      0.69      0.62     15487

    accuracy                           0.71     44056
   macro avg       0.69      0.70      0.69     44056
weighted avg       0.72      0.71      0.71     44056

